In [2]:
import pandas as pd

# -------------------------------
# DATEIEN
# -------------------------------
excel_file = r"C:\Users\sickerti\Projekte\InhaltsverzeichnisseKI\Ki_juni_2026_2.xlsx"
dat_file = r"C:\Users\sickerti\Projekte\inhaltsverzeichnis\I_Daten.dat"

# -------------------------------
# EXCEL LADEN
# -------------------------------
df = pd.read_excel(excel_file)

if "IDN" not in df.columns:
    raise ValueError("Spalte 'IDN' fehlt in Excel")

idn_list = df["IDN"].astype(str).str.strip().tolist()

# -------------------------------
# SPALTEN ANLEGEN (falls nicht vorhanden)
# -------------------------------
if "hat_045P" not in df.columns:
    df["hat_045P"] = None

if "inhalt_045P" not in df.columns:
    df["inhalt_045P"] = None

# -------------------------------
# PICA HELPERS
# -------------------------------
def extract_subfield(record, field_tag, subfield):
    fields = record.split("\x1e")
    for field in fields:
        field = field.strip()
        if field.startswith(field_tag):
            subfields = field.split("\x1f")
            for sf in subfields[1:]:
                if sf.startswith(subfield):
                    return sf[len(subfield):].strip()
    return None


def iter_records(file_path):
    buffer = ""
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if line.startswith("001@") and buffer:
                yield buffer
                buffer = line
            else:
                buffer += line
        if buffer:
            yield buffer

# -------------------------------
# INDEX BAUEN (schneller Lookup)
# -------------------------------
print("🔍 Indexiere I_Daten.dat ...")

index = {}

for rec in iter_records(dat_file):
    idn = extract_subfield(rec, "003@", "0")
    if not idn:
        continue

    if idn in idn_list:
        value = extract_subfield(rec, "045P", "a")
        index[idn] = {
            "has": value is not None,
            "value": value
        }

# -------------------------------
# EXCEL ANREICHERN
# -------------------------------
print("✏️ Schreibe Daten in Excel ...")

for i, row in df.iterrows():
    idn = str(row["IDN"]).strip()

    if idn in index:
        df.at[i, "hat_045P"] = index[idn]["has"]
        df.at[i, "inhalt_045P"] = index[idn]["value"]
    else:
        df.at[i, "hat_045P"] = False
        df.at[i, "inhalt_045P"] = None

# -------------------------------
# SPEICHERN (ÜBERSCHREIBT ORIGINAL)
# -------------------------------
df.to_excel(excel_file, index=False)

print(f"✅ Fertig! Excel erweitert: {excel_file}")

🔍 Indexiere I_Daten.dat ...
✏️ Schreibe Daten in Excel ...
✅ Fertig! Excel erweitert: C:\Users\sickerti\Projekte\InhaltsverzeichnisseKI\Ki_juni_2026_2.xlsx
